# Semiconductor optical amplifiers and superluminescent LEDs {#sec-soa}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter4/notebooks/soa-sled.ipynb){.colab-link} &nbsp;Run this chapter's code in Google Colab (no installation).

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter4/notebooks/soa-sled.ipynb)

> Generated from the Quarto companion. The browser sliders of the website are not included; see `interactive_explorers.ipynb` for ipywidgets versions.

In [ ]:
# --- setup: works locally and in Google Colab ---------------------------
import sys
if 'google.colab' in sys.modules:
    import urllib.request
    urllib.request.urlretrieve('https://raw.githubusercontent.com/philippevelha/Optoelectronics-teaching/main/chapter4/laserlib.py', 'laserlib.py')
else:
    sys.path.insert(0, '..')                 # laserlib.py is in the parent folder
def ojs_define(**kw): pass                   # only used by the Quarto website
# ----------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import laserlib as lz
lz.use_style()

*Slides 152–158 (semiconductor optical amplifiers: travelling-wave and Fabry–Perot types, gain saturation, polarisation-dependent
gain, commercial devices) and 159–161 (superluminescent LEDs).*

## Key relations

A laser diode below threshold is an amplifier: light that enters one facet leaves the other amplified by the single-pass gain
$G_s=e^{(\Gamma g-\alpha_s)L}$. Two versions exist (slides 155–157):

* The **travelling-wave (TW) SOA** has anti-reflection coated (and often angled) facets: one pass, broad and smooth gain
  spectrum (30–60 nm), about 20–30 dB.
* The **Fabry–Perot (FP) SOA** keeps reflecting facets: a regenerative amplifier, whose gain is enhanced at the cavity resonances
  and suppressed between them,
  $$G_{FP}=\frac{(1-R_1)(1-R_2)G_s}{\left(1-\sqrt{R_1R_2}\,G_s\right)^2+4\sqrt{R_1R_2}\,G_s\sin^2\phi},$$
  with $\phi$ the single-pass phase. The gain ripple is $\left[(1+\sqrt{R_1R_2}G_s)/(1-\sqrt{R_1R_2}G_s)\right]^2$.

The gain **saturates** when the signal depletes the carriers (slide 155c). For a homogeneously broadened medium,
$G=G_0\exp[-(G-1)P_\text{in}/P_\text{sat}]$, and the output at which the gain has dropped by 3 dB, $P_{o,\text{sat}}$, is
quoted in datasheets (slide 158). Because the carrier lifetime is ~0.1–1 ns, the SOA gain follows the bit pattern, unlike the
EDFA (@sec-edfa): this causes cross-talk between WDM channels, but also enables all-optical wavelength conversion.

A **superluminescent LED** (SLED, slides 159–161) is an SOA with no input: spontaneous emission generated at one end is amplified on
its way out, with feedback suppressed by angled facets, an absorbing section or AR coatings. The output is bright and directional
like a laser but broadband and incoherent like an LED.

## Fabry–Perot versus travelling-wave SOA

```{ojs}
//| echo: false
viewof sa = Inputs.form({
  GsdB: Inputs.range([0, 35], {value: 25, step: 0.5, label: "single-pass gain G_s (dB)"}),
  lR: Inputs.range([-6, -0.5], {value: -3, step: 0.05, label: "log₁₀ facet reflectance R₁ = R₂"}),
  L: Inputs.range([200, 2000], {value: 500, step: 50, label: "chip length (µm), n_g = 3.6"})
})
```

```{ojs}
//| echo: false
saOut = {
  const R = 10 ** sa.lR, Gs = 10 ** (sa.GsdB/10), rr = R*Gs, fsr = 299792458/(2*3.6*sa.L*1e-6), data = [];
  for (let i = 0; i <= 1200; i++) { const f = -2*fsr + 4*fsr*i/1200, phi = Math.PI*f/fsr;
    data.push({f: f/1e9, G: 10*Math.log10((1 - R)**2*Gs/((1 - rr)**2 + 4*rr*Math.sin(phi)**2))}); }
  return {data, R, rr, fsr, ripple: rr < 1 ? 20*Math.log10((1 + rr)/(1 - rr)) : Infinity};
}
Plot.plot({
  width: Math.min(width, 1050), height: 280, x: {label: "frequency offset (GHz)"}, y: {label: "amplifier gain (dB)", grid: true},
  marks: [Plot.line(saOut.data, {x: "f", y: "G", stroke: "#2a78d6", strokeWidth: 2}), Plot.ruleY([sa.GsdB], {stroke: "#eb6834", strokeDasharray: "4,3"})]
})
```

```{ojs}
//| echo: false
html`<div class="result-box">R G<sub>s</sub> = <b>${saOut.rr.toPrecision(3)}</b> ${saOut.rr >= 1 ? html`≥ 1: <b>the device lases</b> (not an amplifier)` :
html`· gain ripple <b>${saOut.ripple.toFixed(2)} dB</b> · resonance spacing ${(saOut.fsr/1e9).toFixed(1)} GHz · orange: travelling-wave gain G<sub>s</sub>`}</div>`
```

With cleaved facets ($R=0.32$) the chip lases as soon as $G_s>3$ (5 dB). A 25 dB TW amplifier needs $R\lesssim10^{-4}$ for a
ripple below 0.5 dB, which no single AR coating achieves: practical SOAs combine AR coatings with angled waveguides (7–10°) and
window regions.

In [ ]:
#| label: fig-ripple
#| fig-cap: "Peak-to-valley gain ripple of an SOA versus facet reflectance for three single-pass gains. The vertical band marks good AR coatings (10⁻⁴–10⁻³)."
R = np.logspace(-6, -1, 300)
fig, ax = plt.subplots(figsize=(7.2, 3.5))
for GdB in (15, 20, 25, 30):
    rr = R*10**(GdB/10)
    ax.semilogx(R, np.where(rr < 1, 20*np.log10((1 + rr)/(1 - rr)), np.nan), label=f"G_s = {GdB} dB")
ax.axvspan(1e-4, 1e-3, color=lz.GRID, alpha=0.6, zorder=0)
ax.set(xlabel="facet reflectance R", ylabel="gain ripple (dB)", ylim=(0, 15)); ax.legend(fontsize=8); plt.show()

## Gain saturation (slide 155c)

In [ ]:
#| label: fig-soa-sat
#| fig-cap: "Gain versus output power of a TW SOA with small-signal gains of 20 and 25 dB (P_sat = 10 mW of the medium), the shape of slide 155c. The circles mark the 3 dB saturated output power P_o,sat ≈ P_sat ln2 · G₀/(G₀ − 2)."
Pin = np.logspace(-7, -1, 400)
fig, ax = plt.subplots(figsize=(7.2, 3.6))
for G0dB, c in ((20, lz.PALETTE[0]), (25, lz.PALETTE[1])):
    G0 = 10**(G0dB/10)
    G = lz.saturated_gain(G0, Pin, 10e-3)
    Pout = G*Pin
    ax.plot(lz.dB(Pout/1e-3), lz.dB(G), color=c, label=f"G₀ = {G0dB} dB")
    i = np.argmin(abs(lz.dB(G) - (G0dB - 3)))
    ax.plot(lz.dB(Pout[i]/1e-3), lz.dB(G[i]), "o", color=c)
    print(f"G0 = {G0dB} dB: P_o,sat = {lz.dB(Pout[i]/1e-3):.1f} dBm (formula {lz.dB(10*np.log(2)*G0/(G0 - 2)):.1f} dBm)")
ax.set(xlabel="output power (dBm)", ylabel="gain (dB)", xlim=(-20, 20)); ax.legend(fontsize=8); plt.show()

The saturated output power hardly depends on the small-signal gain: it is a property of the medium,
$P_\text{sat}=h\nu A/(\Gamma a\tau)$ with $a$ the differential gain and $\tau$ the carrier lifetime. A large mode area and a
short lifetime raise it; the 8–11 dBm of slide 158 is typical.

In [ ]:
#| label: tbl-soa
#| tbl-cap: "Slide 158: commercial TW SOAs. The last column converts the bandwidth to THz."
s = pd.DataFrame([(850, 25, 40, 150, 2.2, 8, None, 7), (1310, 22, 45, 250, 2, 10, 7, 0.5), (1550, 20, 35, 200, 2, 11, 6, 0.5)],
                 columns=["λ (nm)", "gain (dB)", "BW (nm)", "I_F (mA)", "V_F (V)", "P_o,sat (dBm)", "NF (dB)", "ΔG_p (dB)"])
s["BW (THz)"] = (lz.c0*s["BW (nm)"]*1e-9/(s["λ (nm)"]*1e-9)**2/1e12).round(1)
s["electrical power (mW)"] = s["I_F (mA)"]*s["V_F (V)"]
s

SOAs have NF ≈ 6–7 dB, worse than EDFAs (inversion is never complete, and the input coupling loss adds directly to the NF), and
they need hundreds of mW of electrical power. Their advantages are size, any wavelength from 650 to 1650 nm, and integration with
lasers and modulators on InP chips.

## Superluminescent LEDs (slides 159–161)

The spontaneous emission generated near the rear of the stripe is amplified along it. Since the gain is largest at the peak, the
output spectrum narrows as the gain grows (**gain narrowing**), but it stays far broader than a laser line.

In [ ]:
#| label: fig-sled
#| fig-cap: "Output spectrum of a 1310 nm SLED model: spontaneous emission (Gaussian, 70 nm FWHM) amplified by a gain spectrum of the same shape, P(λ) ∝ r_sp(λ)[G(λ) − 1]/g(λ). Increasing the peak single-pass gain narrows the spectrum; the coherence length λ²/Δλ, which sets the axial resolution of optical coherence tomography, grows accordingly."
lam = np.linspace(1150e-9, 1470e-9, 800)
shape = np.exp(-4*np.log(2)*((lam - 1310e-9)/70e-9)**2)
fig, ax = plt.subplots(figsize=(7.5, 3.6))
for GdB in (0.1, 10, 20, 30):
    gL = GdB/4.343*shape
    P = shape*np.expm1(gL)/np.maximum(gL, 1e-9)
    P /= P.max()
    w = lam[P >= 0.5]; fw = w[-1] - w[0]
    ax.plot(lam*1e9, P, label=f"peak gain {GdB:g} dB: FWHM {fw*1e9:.0f} nm, ℓ_c ≈ {1310e-9**2/fw*1e6:.0f} µm")
ax.set(xlabel="wavelength (nm)", ylabel="normalised power"); ax.legend(fontsize=8); plt.show()

SLEDs are the light source of fiber-optic gyroscopes and of optical coherence tomography (OCT), where the coherence length
$\ell_c\approx\lambda^2/\Delta\lambda$ (Ch. 1) sets the depth resolution: about 10–20 µm for the 40–45 nm SLEDs of slide 159.

::: {.callout-tip collapse="true"}
## Going further
- **Pattern effects and wavelength conversion.** Solve $dn/dt=I/eV-n/\tau-\Gamma a(n-n_{tr})P/(h\nu A)$ for a 10 Gb/s on-off signal
  through an SOA with $\tau=200$ ps. The gain recovers too slowly: the "1" bits after a run of "0"s are amplified more. A continuous
  probe at another wavelength receives the inverted pattern: this is cross-gain-modulation wavelength conversion.
- **Polarisation-dependent gain** (slide 157). Bulk active layers with a nearly square cross section, or tensile-strained QWs that
  add TM gain, bring $\Delta G_p$ below 0.5 dB (slide 158).
- **SOA versus EDFA noise.** Add the input coupling loss (3 dB from fiber to chip) to the intrinsic $2n_{sp}$ with the Friis formula
  of @sec-edfa-noise, and show why the SOA NF rarely goes below 6 dB.
:::

::: {.callout-note collapse="true"}
## Complements
- M. J. Connelly, *Semiconductor Optical Amplifiers* (Kluwer, 2002).
- Kasap, sec. 4.18–4.19; Agrawal, *Fiber-Optic Communication Systems*, sec. 7.2.
:::

## Try it yourself

1. A 1 mm SOA has a single-pass gain of 30 dB. What facet reflectance keeps the ripple below 1 dB?
2. Using the saturation model, at what input power does a 25 dB SOA with $P_\text{sat}=10$ mW lose 1 dB of gain?
3. Why does a FP SOA have a narrower optical bandwidth (per resonance) than a TW SOA, but a higher peak gain?
4. Estimate the OCT axial resolution (in tissue, $n=1.38$) with a 1310 nm SLED of 45 nm FWHM. What spectral width would give 3 µm?